# RAG-система для аналізу фінансових звітів компаній

Стек: LangChain + Chroma + OpenAI (text-embedding-3-small)
Логіка: 4 компанії → 4 чанки → векторне сховище → retriever (k=2) → LLM

In [5]:
# ==========================================================
# 1. Імпорти та вхідні дані
# ==========================================================


import os
import sys
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv()  

# Фінансові дані компаній за 2024 рік.
# Кожна компанія відділена ПУСТИМ РЯДКОМ (\n\n) — це і є наш роздільник.
finansovi_dani = """
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу.
""".strip()

In [6]:
# ==========================================================
# 2. Розбиття на чанки
# ==========================================================

# Роздільник ["\n\n"] + достатньо великий chunk_size, щоб кожна компанія залишилася ЦІЛИМ чанком і не була розрізана посередині.

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=400,
    chunk_overlap=0,
)

chunks = text_splitter.create_documents([finansovi_dani])

print(f"Кількість чанків: {len(chunks)}")
for i, chunk in enumerate(chunks, 1):
    print(f"\n--- Чанк {i} ---\n{chunk.page_content[:80]}...")

Кількість чанків: 4

--- Чанк 1 ---
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67...

--- Чанк 2 ---
ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий при...

--- Чанк 3 ---
РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуто...

--- Чанк 4 ---
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток ...


In [7]:
# ==========================================================
# 3. Векторне сховище та метадані
# ==========================================================

# Ключ OpenRouter береться з .env (OPENROUTER_API_KEY)
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
)

# Додаємо осмислені метадані: назва компанії для кожного чанка
nazvy_kompaniy = ["ТехНова Інк", "ГрінЕнерджі Солюшнс", "РітейлМакс Груп", "ФінТех Дайнамікс"]
for chunk, nazva in zip(chunks, nazvy_kompaniy):
    chunk.metadata = {"kompaniya": nazva, "rik": 2024}

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="finansovi_zvity_kompaniy",
    collection_metadata={"opys": "Фінансові звіти компаній за 2024 рік"},
)

print("Документів у колекції:", vectorstore._collection.count())

Документів у колекції: 4


In [8]:
rezultaty = vectorstore.similarity_search("Яка виручка у ТехНова Інк?", k=2)
for r in rezultaty:
    print(r.metadata["kompaniya"], "->", r.page_content[:60], "...")

ТехНова Інк -> ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків) ...
ФінТех Дайнамікс -> ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсо ...


In [9]:
# ==========================================================
# 4. Retriever та LLM
# ==========================================================

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

# Перевірка: що знаходить retriever
docs = retriever.invoke("Скільки працівників у ГрінЕнерджі Солюшнс?")
print("Знайдено чанків:", len(docs))
for d in docs:
    print("-", d.metadata["kompaniya"])


Знайдено чанків: 2
- ГрінЕнерджі Солюшнс
- ТехНова Інк


In [10]:
pytannya = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
]
for p in pytannya:
    docs = vectorstore.similarity_search(p, k=2)
    print(p, "->", [d.metadata["kompaniya"] for d in docs])

Яка виручка у компанії ТехНова Інк за 2024 рік? -> ['ТехНова Інк', 'ФінТех Дайнамікс']
Скільки працівників у ГрінЕнерджі Солюшнс? -> ['ГрінЕнерджі Солюшнс', 'ТехНова Інк']
Який показник ROE у РітейлМакс Груп? -> ['РітейлМакс Груп', 'ФінТех Дайнамікс']
Де знаходиться штаб-квартира ТехНова Інк? -> ['ТехНова Інк', 'ФінТех Дайнамікс']
Яка маржа EBITDA у ФінТех Дайнамікс? -> ['ФінТех Дайнамікс', 'ТехНова Інк']


In [11]:
# ==========================================================
# 5. RAG-ланцюг
# ==========================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    """Ти — фінансовий аналітик. Відповідай українською мовою, коротко і точно.
Використовуй ЛИШЕ наведений нижче контекст. Відповідай тільки про ту компанію,
яку названо в питанні, і обов'язково наводь конкретні цифри з контексту.
Якщо назву компанії з питання в контексті немає, відповідай:
"На основі наданого контексту, інформація про [назва компанії] відсутня."
Нічого не вигадуй.

Контекст:
{context}

Питання: {question}

Відповідь:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Швидка перевірка одного питання
print(rag_chain.invoke("Яка виручка у компанії ТехНова Інк за 2024 рік?"))

Виручка компанії ТехНова Інк за 2024 рік становить 450 млн дол.


In [12]:
# ==========================================================
# 6. Тестові питання
# ==========================================================

pytannya = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?",
]

print("=" * 70)
for i, p in enumerate(pytannya, 1):
    vidpovid = rag_chain.invoke(p)
    print(f"\nПитання {i}: {p}")
    print(f"Відповідь: {vidpovid}")
    print("-" * 70)


Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------

Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
----------------------------------------------------------------------

Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------

Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------

Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс становить мінус 8 відсотків.
----------------------------------------------------------------------

Питання 6: Який чистий прибуток у ЕкоПакування Солюшнс?
Відпові